# Hierarchical Clustering - UPGMA

---
## Learning Objectives

1. Implement a distance-based phylogeny neighbor joining algorithm
1. Understanding of hierarchical clustering
1. Manipulation of arrays and lists in Python


---
## Imports
You will need to install a two new packages to properly render your phylogenetic trees. Using your preferred package manager, install both [`biopython`](https://biopython.org/) and `matplotlib` into your virtual environments.

For example:
```bash
$ micromamba create -n module06 -c bioconda -c conda-forge python numpy matplotlib biopython
```

In [ ]:
from typing import List, Tuple, Dict
from io import StringIO

import matplotlib.pyplot as plt
from Bio import Phylo
import numpy as np

---
## Background

Today we will implement a distance-based phylogenetic tree construction method using the Neighbor-Joining (NJ) algorithm and Smith-Waterman local alignment scores. Unlike UPGMA (Unweighted Pair Group Method using Arithmetic averages), NJ produces unrooted trees and does not assume a constant evolutionary rate across lineages, making it more biologically realistic for analyzing sequence relationships.

We will use HIV-1 reverse transcriptase sequences to construct our phylogenetic tree. The Smith-Waterman algorithm will be used to generate pairwise local alignment scores, which will then be converted into distances for tree construction. This approach is particularly suitable for HIV sequence analysis as it:
1. Handles sequence variations effectively through local alignment
2. Accounts for potential rate heterogeneity across different viral strains
3. Does not assume a molecular clock

The ultimate output will be an unrooted phylogenetic tree representing the evolutionary relationships between the HIV-1 sequences, visualized using the ete3 library. This method provides insights into viral diversity and evolutionary patterns while avoiding the assumptions of simpler hierarchical clustering approaches.

The key innovations of this implementation are:
- Use of Smith-Waterman for sensitive local alignment scoring
- Implementation of Neighbor-Joining for unrooted tree construction
- More biologically realistic evolutionary model

## Neighbor-Joining Algorithm

**Input**: Distance matrix $D$ for $n$ sequences

**Initialization**:
- Let $n$ be the number of sequences
- Assign each sequence $i$ to its own leaf node
- Initialize tree $T$ with leaf nodes

**Iteration**:
While $n > 2$:
1. Calculate Q-matrix where:
   $Q(i,j) = (n-2)d(i,j) - \sum_{k=1}^n d(i,k) - \sum_{k=1}^n d(j,k)$

2. Find pair $(i,j)$ with minimum $Q(i,j)$

3. Calculate branch lengths:
   $d_i = \frac{d(i,j) + (r_i - r_j)/(n-2)}{2}$
   $d_j = d(i,j) - d_i$
   where $r_i = \sum_{k=1}^n d(i,k)$

4. Create new node $k$
   - Add branches from $k$ to $i$ and $j$ with lengths $d_i$ and $d_j$

5. Update distances to remaining nodes $x$:
   $d(k,x) = \frac{d(i,x) + d(j,x) - d(i,j)}{2}$

6. Remove nodes $i$ and $j$
7. Add node $k$ to active nodes
8. $n = n - 1$

**Termination**:
When $n = 2$ with remaining nodes $i$ and $j$:
- Add final branch between $i$ and $j$ with length $d(i,j)$
- Return unrooted tree $T$

---
## Distance metrics for comparing sequences

### Smith-Waterman
This was previously implemented. Feel free to import if you know how...or Copy & Paste.



In [ ]:
import numpy as np

In [ ]:
def read_fasta(filename: str) -> Dict[str, str]:
    """Reads sequences from FASTA file.

    Args:
        filename (str): Path to FASTA file containing HIV RT sequences

    Returns:
        Dict[str, str]: Dictionary mapping sequence IDs to sequences

    Examples:
        >>> seqs = read_fasta("lafayette_SARS_RT.fasta")
        >>> len(seqs) > 0
        True
    """
    pass

In [ ]:
# Global variables for directionality
END = 0
DIAG = 1
UP = 2
LEFT = 3

In [ ]:
def cal_score(matrix, seq1, seq2, i, j, match, mismatch, gap):
    '''Calculate score for position (i,j) in scoring matrix, also record move to trace back

    Args:
        matrix (numpy array): scoring matrix
        seq1 (str): sequence 1
        seq2 (str): sequence 2
        i (int): current row number
        j (int): current column number

    Returns:
        score in position (i,j)
        move to trace back: 0-END, 1-DIAG, 2-UP, 3-LEFT


    FROM PROJECT 5
    Pseudocode:
        Compare sequence 1 and 2
        Define the match/mismatch/gap scores
            Note: negatives scores will be converted to zeroes

        Calculate scores based on upper-left, up, and left neighbors:
            diag_score = upper-left + (match or mismatch)
            up_score = up + gap
            left_score = left + gap
        score = max(0, diag_score, up_score, left_score)
        traceback = maximum direction or end

        figure out the best score
        maximum of 0, diag_score, up_score, left_score

        figure out traceback direction
        if the best score is 0
            end
        else:
            if the best score != 0
                up diagonal or left
        Return the traceback direction and score
    '''


    if seq1[i - 1] == seq2[j - 1]:
        diag_score = matrix[i - 1][j - 1] + match
    else:
        diag_score = matrix[i - 1][j - 1] + mismatch

    # calculate the gap scores
    up_score = matrix[i - 1][j] + gap
    left_score = matrix[i][j - 1] + gap

    # Calculate the max of the cell, 0 ensures no negatives
    best_score = max(0, diag_score, up_score, left_score)

    # Establish positions; tie-breaker priority: diag --> up --> left
    if best_score == 0:
        direction = END
    elif best_score == diag_score:
        direction = DIAG
    elif best_score == up_score:
        direction = UP
    else:
        direction = LEFT


    return best_score, direction


In [ ]:
def traceback(seq1, seq2, traceback_matrix, maximum_position):
    '''
    Find the optimal path through scoring matrix

        diagonal: match/mismatch
        up: gap in seq1
        left: gap in seq2

    Args:
        seq1 (str) : First sequence being aligned
        seq2 (str) : Second sequence being aligned
        traceback_matrix (numpy array): traceback matrix
        maximum_position (tuple): starting position to trace back from

    Returns:
        aligned_seq1 (str): e.g. GTTGAC
        aligned_seq2 (str): e.g. GTT-AC


    FROM PROJECT 5
    Pseudocode:
        while current_move != END:
            current_move = traceback_matrix[current_row][current_col]
            if current_move == DIAG:
                ...
            elif current_move == UP:
                ...
            elif current_move == LEFT:
                ...
    '''

    #initialize empty lists to hold the aligned sequences
    traceback_seq1 = []
    traceback_seq2 = []

    # Start at the max position
    current_row, current_col = maximum_position

    # While loop basic structure given by professor pseudocode, modified by us
    # While it's true that our current move isn't the end, our current move is in the matrix
    while True:
        current_move = traceback_matrix[current_row][current_col]

        # if our current move is end, while loop condition becomes false, break loop
        if current_move == END:
            break

        # Basic structure note: aligned_seq1 tracks rows, aligned_seq2 tracks columns
        # moving up adds a gap in the column, so you wouldn't decrement current_col
            # And you don't append the column bp, it's a gap
        # moving left adds a gap in the row, so you wouldn't decrement current_row
            # And you don't append the row bp, it's a gap

        # If current move is diagonal, we append one bp and decrease row/column by 1
        if current_move == DIAG:
            traceback_seq1.append(seq1[current_row - 1]) #append both row and column
            traceback_seq2.append(seq2[current_col - 1]) # subtract 1 from these so we don't go out of bounds

            current_row = current_row - 1 # decrease row and column by 1 to move on
            current_col = current_col - 1

         # If current move is up, that creates a gap in the columns sequence
        elif current_move == UP:
            traceback_seq1.append(seq1[current_row - 1]) # append only the row
            traceback_seq2.append("-") # append "-" into aligned_seq2 (columns)

            current_row = current_row - 1 # decrease row count to move on


        # If current move is left, that creates a gap in the rows sequence
        elif current_move == LEFT:
            traceback_seq2.append(seq2[current_col - 1]) # append only the column
            traceback_seq1.append("-") # append "-" into aligned_seq1 (rows)

            current_col = current_col - 1 # decrease column count to move on

    # reverse and join the sequences for alignment
    aligned_seq1 = "".join(reversed(traceback_seq1))
    aligned_seq2 = "".join(reversed(traceback_seq2))

    return aligned_seq1, aligned_seq2



In [ ]:
import numpy as np
def prev_smith_waterman(seq1, seq2, match=2.0, mismatch=-1.0, gap=-1.0):
    '''Smith-Waterman algorithm for local alignment

    Args:
        seq1 (str): input seq 1
        seq2 (str): input seq 2
        match: default = +2.0
        mismatch: default = -1.0
        gap: default = -1.0

    Returns:
        aligned_seq1 (str)
        aligned_seq2 (str)
        score_matrix (numpy array): scoring matrix

    Pseudocode:
    Initialize scoring matrix H with zeros
    For i from 1 to m:
        For j from 1 to n:
            H[i][j] = max {
                0
                H[i-1][j-1] + match/mismatch score
                H[i-1][j] + gap penalty
                H[i][j-1] + gap penalty
                }
    Find the highest score in  H text and its position
    Perform traceback from the highest score position
    Return the optimal local alignment

    # NOTE: THIS IS PASTED FROM PROJECT 5, RENAMED, AND RETURN SIGNATURE MODIFIED.

    '''

    # initialize sequence lengths m and n
    m, n = len(seq1), len(seq2)

    # initialize the base maximum score and positions for i and j
    highest_score = 0
    highest_position = (0,0)

    # initialize scoring matrix H with zeros using numpy, add one because S-W uses an extra row/column
    score_matrix = np.zeros((m+1, n+1), dtype=int)
    traceback_matrix = np.zeros((m+1, n+1))

    # filling out scoring matrix H
    for i in range(1, m + 1):
        for j in range(1, n + 1):

            # call cal_score on our variables
            best_score, direction = cal_score(score_matrix, seq1, seq2, i, j, match, mismatch, gap)

            # assign the best score into the matrix
            score_matrix[i][j] = best_score
            traceback_matrix[i][j] = direction

            # Check scoring matrix H against the maximum score and positioning
            if best_score > highest_score:
                highest_score = best_score
                highest_position = (i, j)

    # call traceback for the aligned sequences
    #aligned_seq1, aligned_seq2 = traceback(seq1, seq2, traceback_matrix, highest_position)

    return highest_score

In [ ]:
import math
def smith_waterman(
    sequence1: str,
    sequence2: str,
    match_score: float = 2.0,
    mismatch_penalty: float = -1.0,
    gap_penalty: float = -1.0
) -> float:
    """Implements Smith-Waterman local alignment algorithm.

    Args:
        sequence1 (str): First HIV RT sequence
        sequence2 (str): Second HIV RT sequence
        match_score (float): Score for matching characters
        mismatch_penalty (float): Penalty for mismatched characters
        gap_penalty (float): Penalty for gaps

    Returns:
        float: Normalized similarity score between sequences

    Examples:
        >>> seq1 = "ACGT"
        >>> seq2 = "ACGT"
        >>> smith_waterman(seq1, seq2)
        1.0

    pseudocode:
    call previous smith waterman algorithm from project 5 to get:
    self score of sequence 1
    self score of sequence 2
    highest score comparison between sequence 1 and 2

    Normalization step:
    if the self scores are not 0
        S(i,j) = SW(i,j) / sqrt(SW(i,i) x SW(j,j))
    else if there is a zero in the denominator
        only return a 0.0 so the program doesnt break

    """

    # raw local alignment score of (i,i) by calling previous SW
    self_scorei = prev_smith_waterman(sequence1, sequence1, match_score, mismatch_penalty, gap_penalty)

    # raw local alignment score of (j,j) by calling previous SW
    self_scorej = prev_smith_waterman(sequence2, sequence2, match_score, mismatch_penalty, gap_penalty)

    # maximum alignment score of (i,j) by calling previous SW
    sequence_comparison = prev_smith_waterman(sequence1, sequence2, match_score, mismatch_penalty, gap_penalty)

    # Make sure the denominator isn't 0
    if self_scorei != 0 and self_scorej != 0:
        # calculate normalized score by dividing the max (i,j) by sqrt of (i,i)x(jj)
        norm_score = sequence_comparison / (math.sqrt(self_scorei * self_scorej))

    else:
        # if the denominator is 0, just break and return 0.0 for no similarities
        return 0.0

    return norm_score

In [ ]:
def build_distance_matrix(sequences: Dict[str, str]) -> Tuple[np.ndarray, List[str]]:
    """Builds DISTANCE (not similiarity) matrix using Smith-Waterman scores.

    Args:
        sequences (Dict[str, str]): Dictionary of HIV RT sequences

    Returns:
        Tuple[np.ndarray, List[str]]: Distance matrix and list of sequence IDs

    Examples:
        >>> seqs = read_fasta("lafayette_SARS_RT.fasta")
        >>> mat, ids = build_distance_matrix(seqs)
        >>> mat[0:3, 0:3]  # Show 3x3 slice of distance matrix
        array([[0.000, 0.004, 0.012],
              [0.004, 0.000, 0.012],
              [0.012, 0.012, 0.000]])
        >>> len(ids)  # Number of sequences
        19

    Normalized scores need to be converted to distance in BDM():
    D(i,j) = 1 - S(i,j)

    pseudocode:
    extract sequence names by thieir index
    initialize a distance matrix of zeros

    """
    sequence_names = list(sequences.keys())
    distance_matrix = np.zeros((len(sequences), len(sequences)))

    for i in range(0, len(sequences)-1):
        for j in range(i+1, len(sequences)):
            sequence_i = sequences[sequence_names[i]]
            sequence_j = sequences[sequence_names[j]]
            score_conv = smith_waterman(sequence_i, sequence_j)
            distance_scores = 1-score_conv
            distance_matrix[i, j] = distance_scores
            distance_matrix[j, i] = distance_scores


    return distance_matrix, sequence_names

In [ ]:
def neighbor_joining(
    distance_matrix: np.ndarray,
    labels: List[str]
) -> str:
    """Implements Neighbor-Joining algorithm for phylogenetic tree construction.

    Args:
        distance_matrix (np.ndarray): Distance matrix from Smith-Waterman scores
        labels (List[str]): Sequence identifiers

    Returns:
        str: Newick format tree string

    Examples:
        >>> seqs = read_fasta("lafayette_SARS_RT.fasta")
        >>> mat, ids = build_distance_matrix(seqs)
        >>> tree = neighbor_joining(mat, ids)
        >>> tree.startswith("((DM1:0.002,DM2:0.002)")  # Start of Newick string
        True
        >>> tree.count(",")  # Number of separators in tree
        18

    Logic Flow:
    Calculate row totals.
    Calculate (D*) for every distinct pair.
    Find the pair with the smallest (D*).
    Calculate its two branch lengths.
    Create a new internal node and record its connections.
    Calculate distances from the new node to all remaining nodes.
    Update the matrix and repeat until the stopping condition is reached.

    At the final joining n=3:
        L(a)=1/2(D(a,b)+D(a,c)−D(b,c))
        L(b)=1/2(D(a,b)+D(b,c)−D(a,c))
        L(c)=1/2(D(a,c)+D(b,c)−D(a,b))

    """



    pass

In [ ]:
def neighbor_join_recursion(distance_matrix: np.ndarray, labels: List[str], internal_node_count) -> str:
    """
    Helper function for neighbor-joining recursion.

    :return:
    """
    row_totals = np.sum(distance_matrix, axis=1)
    current_nodes = distance_matrix.shape[0]

    if current_nodes > 3:
        minimum_dstar = float('inf')
        best_pair = (0,0)
        for i in range(current_nodes-1):
            for j in range(i+1, current_nodes):
                d_star = (current_nodes - 2) * distance_matrix[i][j] - row_totals[i] - row_totals[j]
                if d_star < minimum_dstar:
                    minimum_dstar = d_star
                    best_pair = (i,j)
        i, j = best_pair
        delta = (row_totals[i] - row_totals[j]) / (current_nodes - 2)
        branch_length_i = (distance_matrix[i][j] + delta) / 2
        branch_length_j = distance_matrix[i][j] - branch_length_i

        # Still need to keep track of what gets removed and their branch lengths for the final return

        new_distances = []
        for k in range(current_nodes):
            if k != i and k != j:
                new_distance = (distance_matrix[i][k] + distance_matrix[j][k] - distance_matrix[i][j]) / 2
                new_distances.append(new_distance)

        smaller_matrix = np.delete(np.delete(distance_matrix, [i, j], axis=0), [i, j], axis=1)
        smaller_matrix = np.insert(smaller_matrix, len(smaller_matrix), new_distances, axis=0)
        smaller_matrix = np.insert(smaller_matrix, len(smaller_matrix), [*new_distances, 0], axis=1)



        # Need to redo entire internal node counter
        #internal_nodes = len(sequences) - current_nodes + 1
        #updated_node_name = f"U{internal_nodes}"

        # need to remove labels for i and j


    elif current_nodes == 3:
        # Get the distance between each of the remaining 3 nodes
        ab_distance = distance_matrix[0][1]
        ac_distance = distance_matrix[0][2]
        bc_distance = distance_matrix[1][2]

        # L(a)=1/2(D(a,b)+D(a,c)−D(b,c))
        branch_length_a = (1/2) * (ab_distance + ac_distance - bc_distance)
        # L(b)=1/2(D(a,b)+D(b,c)−D(a,c))
        branch_length_b = (1/2) * (ab_distance + bc_distance - ac_distance)
         # L(c)=1/2(D(a,c)+D(b,c)−D(a,b))
        branch_length_c = (1/2) * (ac_distance + bc_distance - ab_distance)

        #


In [ ]:
def plot_tree(newick_tree: str) -> None:
    """Plots a phylogenetic tree from a Newick string using Biopython and Matplotlib.

    The tree is rendered in a rectangular layout with smaller leaf labels and
    added margins to reduce overlap between labels and branches.

    Args:
        newick_tree (str): Tree in Newick format

    Returns:
        None: Displays the plotted tree
    """
    handle = StringIO(newick_tree)
    tree = Phylo.read(handle, "newick")

    fig, ax = plt.subplots(figsize=(8, 10))

    Phylo.draw(
        tree,
        axes=ax,
        do_show=False,
        label_func=lambda clade: clade.name if clade.is_terminal() else None,
    )
    for text in ax.texts:
        text.set_fontsize(6)              # shrink labels
    ax.margins(x=0.1, y=0.05)             # extra padding around tree

    plt.tight_layout()
    plt.show()


In [ ]:
if __name__ == "__main__":
    # Read HIV RT sequences
    sequences = read_fasta("lafayette_SARS_RT.fasta")
    
    # Build distance matrix using Smith-Waterman
    dist_matrix, seq_ids = build_distance_matrix(sequences)
    
    # Generate unrooted tree using Neighbor-Joining
    tree = neighbor_joining(dist_matrix, seq_ids)
    
    # Plot tree
    plot_tree(tree)